#  Data Cleaning & Preprocessing

## Objective
Transform raw transactional data into a clean, analysis-ready dataset by:
1. **Removing Invalid Records:** Handle missing customer IDs, product returns, and data quality issues
2. **Engineering Base Features:** Calculate transaction-level revenue metrics
3. **Validating Data Integrity:** Ensure all records meet business logic requirements

## The Problem We're Solving
In non-contractual retail settings, raw transaction logs contain:
- **Silent Customers:** Missing IDs make segmentation impossible
- **Returns & Cancellations:** Negative quantities distort spending patterns
- **Invalid Transactions:** Zero/negative prices indicate data errors

**Impact:** Without cleaning, our RFM analysis would misidentify high-value customers and create false behavioral patterns.

In [1]:
import pandas as pd
import os

# Option 1: Read from local file (if already downloaded)
local_path = "../data/raw/online_retail_II.xlsx"

# Option 2: Read directly from UCI repository (original dataset)
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/00352/Online%20Retail.xlsx"

# Try local file first, otherwise download from URL
if os.path.exists(local_path):
    print(" Loading from local file...")
    df = pd.read_excel(local_path)
else:
    print(" Local file not found. Downloading from UCI repository...")
    print("⏳ This may take 1-2 minutes depending on your connection...")
    df = pd.read_excel(url)
    
    # Save for future use
    os.makedirs("../data/raw", exist_ok=True)
    df.to_excel(local_path, index=False)
    print(" Dataset saved locally for future use!")

print(f"\n Successfully loaded {df.shape[0]:,} transactions with {df.shape[1]} columns")
print(f" Date range: {df['InvoiceDate'].min()} to {df['InvoiceDate'].max()}")

 Local file not found. Downloading from UCI repository...
⏳ This may take 1-2 minutes depending on your connection...
 Dataset saved locally for future use!

 Successfully loaded 541,909 transactions with 8 columns
 Date range: 2010-12-01 08:26:00 to 2011-12-09 12:50:00
 Dataset saved locally for future use!

 Successfully loaded 541,909 transactions with 8 columns
 Date range: 2010-12-01 08:26:00 to 2011-12-09 12:50:00


##  Data Cleaning Steps

### Step 1: Remove Missing Customer IDs
**Why?** We cannot perform customer-level segmentation without valid customer identifiers.
- These records represent **anonymous transactions** (e.g., guest checkouts)
- **Business Impact:** Excluding these maintains data integrity for RFM analysis

In [1]:
# Check for data quality issues
print(" Data Quality Issues:")
print(f"\n1. Missing Customer IDs: {df['CustomerID'].isna().sum():,} ({df['CustomerID'].isna().sum()/len(df)*100:.1f}%)")
print(f"2. Missing Descriptions: {df['Description'].isna().sum():,}")
print(f"3. Negative Quantities (Returns): {(df['Quantity'] < 0).sum():,}")
print(f"4. Zero/Negative Prices: {(df['UnitPrice'] <= 0).sum():,}")
print(f"5. Unique Customers: {df['CustomerID'].nunique():,}")
print(f"6. Unique Products: {df['Description'].nunique():,}")

 Data Quality Issues:


NameError: name 'df' is not defined

In [2]:
# Inspect data structure
print(" Dataset Info:")
print(f"   Total Records: {len(df):,}")
print(f"   Columns: {list(df.columns)}")
print(f"\n Sample Data:")
df.head(10)

 Dataset Info:


NameError: name 'df' is not defined

##  Initial Data Exploration
Let's examine the raw data structure and identify quality issues.

In [3]:
print(" BEFORE CLEANING:")
print(f"   Total Records: {len(df):,}")
print(f"   Unique Customers: {df['CustomerID'].nunique():,}")

# Remove missing customer IDs
records_before = len(df)
df = df[df['CustomerID'].notna()]
records_removed = records_before - len(df)

print(f"\n Removed {records_removed:,} records with missing CustomerID ({records_removed/records_before*100:.1f}%)")

# Remove returns and invalid prices
returns_count = (df['Quantity'] <= 0).sum()
df = df[df['Quantity'] > 0]
print(f" Removed {returns_count:,} product returns (negative/zero quantity)")

invalid_price = (df['UnitPrice'] <= 0).sum()
df = df[df['UnitPrice'] > 0]
print(f" Removed {invalid_price:,} records with invalid prices")

print(f"\n AFTER CLEANING:")
print(f"   Total Records: {len(df):,}")
print(f"   Unique Customers: {df['CustomerID'].nunique():,}")
print(f"   Data Retained: {len(df)/records_before*100:.1f}%")

 BEFORE CLEANING:


NameError: name 'df' is not defined

### Step 2: Standardize Customer ID Format
Convert to integer type for consistency and efficient storage.

In [ ]:
df['CustomerID'] = df['CustomerID'].astype(int)
print(f" CustomerID converted to integer format")
print(f"   Sample IDs: {df['CustomerID'].head().tolist()}")

### Step 3: Feature Engineering - Calculate Total Transaction Value
**Business Logic:** `TotalPrice = Quantity × UnitPrice`

This metric represents the **revenue generated per line item** and is critical for calculating the **Monetary** component of RFM.

In [ ]:
df['TotalPrice'] = df['Quantity'] * df['UnitPrice']

print(f" TotalPrice feature created")
print(f"\n Revenue Statistics:")
print(f"   Total Revenue: ${df['TotalPrice'].sum():,.2f}")
print(f"   Average Transaction Value: ${df['TotalPrice'].mean():.2f}")
print(f"   Median Transaction Value: ${df['TotalPrice'].median():.2f}")
print(f"   Max Single Purchase: ${df['TotalPrice'].max():,.2f}")

### Step 4: Final Data Validation
Verify no missing values remain in critical columns.

In [ ]:
print(" Final Missing Value Check:")
missing = df.isnull().sum()
print(missing[missing > 0] if missing.sum() > 0 else " No missing values in critical columns!")

print(f"\n Clean Dataset Summary:")
print(f"   Shape: {df.shape}")
print(f"   Customers: {df['CustomerID'].nunique():,}")
print(f"   Products: {df['Description'].nunique():,}")
print(f"   Invoices: {df['InvoiceNo'].nunique():,}")
print(f"   Countries: {df['Country'].nunique()}")

Invoice        0
StockCode      0
Description    0
Quantity       0
InvoiceDate    0
Price          0
Customer ID    0
Country        0
TotalPrice     0
dtype: int64

##  Save Cleaned Data
Export the cleaned dataset for use in subsequent analysis notebooks.

In [ ]:
os.makedirs("../data/processed", exist_ok=True)
df.to_csv("../data/processed/clean_transactions.csv", index=False)
print(" Clean dataset saved to: data/processed/clean_transactions.csv")
print(f" Ready for RFM feature engineering with {len(df):,} valid transactions!")

---
##  Cleaning Summary

**What We Accomplished:**
1.  Removed ~135,000 records with missing CustomerIDs (anonymous transactions)
2.  Filtered out product returns (negative quantities)
3.  Eliminated invalid pricing data
4.  Engineered TotalPrice revenue metric
5.  Retained ~400,000+ valid customer transactions

**Why This Matters:**
- **Data Integrity:** Clean data ensures accurate RFM calculations
- **Segmentation Quality:** Only identifiable customers can be clustered
- **Revenue Accuracy:** Excluding returns prevents negative monetary values

**Next Step:** → Notebook 02: Feature Engineering (RFM Calculation)